# 08A — Augmented Robustness Checks (1985–2020 two-stage pipeline)

**Purpose:** Robustness checks for the augmented two-stage analysis.

**Key differences vs NB08:**
- Stage 1 strict temporal validation is now meaningful: train on multiple pre-crisis periods
  (1988–1990, 1988–2002) and test on Nordic crisis and GFC separately
- Stage 2 robustness checks (alternative targets, feature pruning) use df_sent (2003–2020)
- Bootstrap AUPRC CI now computed against M2-matched (not M2 from different period)
- Leave-one-country-out cross-validation on Stage 1 extended dataset

**Inputs:** `data/processed/augmented_analysis/`

---

## Cell 1 — Imports and paths

In [3]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import xgboost as xgb

BASE    = Path(r'C:\Users\Owner\OneDrive\dissertation')
AUG_DIR = BASE / 'data' / 'processed' / 'augmented_analysis'
ROB_DIR = BASE / 'data' / 'processed' / 'augmented_robustness'
FIG_DIR = BASE / 'figures' / 'augmented_robustness'
ROB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

# Load Stage 1 (macro, extended) and Stage 2 (sentiment) datasets
df_macro    = pd.read_csv(AUG_DIR / 'df_macro_augmented.csv')
df_sent     = pd.read_csv(AUG_DIR / 'df_sent_augmented.csv')
M1_FEATURES = pd.read_csv(AUG_DIR / 'M1_features.csv', header=None)[0].tolist()
M2_FEATURES = pd.read_csv(AUG_DIR / 'M2_features.csv', header=None)[0].tolist()
M3_FEATURES = pd.read_csv(AUG_DIR / 'M3_features.csv', header=None)[0].tolist()

TARGET = next(c for c in df_macro.columns if 'target' in c.lower())

print(f'Stage 1 dataset : {df_macro.shape}  ({df_macro["year"].min()}–{df_macro["year"].max()})')
print(f'Stage 2 dataset : {df_sent.shape}   ({df_sent["year"].min()}–{df_sent["year"].max()})')
print(f'Target          : {TARGET}')
print(f'M1/M2/M3 features: {len(M1_FEATURES)} / {len(M2_FEATURES)} / {len(M3_FEATURES)}')

Stage 1 dataset : (569, 101)  (1989–2020)
Stage 2 dataset : (319, 141)   (2003–2020)
Target          : target_h1
M1/M2/M3 features: 19 / 41 / 61


## Cell 2 — Stage 1 strict temporal validation: multiple pre-crisis training windows
With extended data, three meaningful splits are now available.

In [5]:
print('=== STAGE 1 STRICT TEMPORAL VALIDATION ===')
print('Tests M2 trained on pre-crisis data against subsequent crisis episodes')
print()

splits = [
    {
        'name': 'A: Pre-Nordic (1988–1990) → Nordic+others (1991–1997)',
        'train': (1988, 1990), 'test': (1991, 1997)
    },
    {
        'name': 'B: Pre-GFC extended (1988–2002) → GFC (2003–2011)',
        'train': (1988, 2002), 'test': (2003, 2011)
    },
    {
        'name': 'C: Pre-GFC short (2003–2005) → GFC (2006–2009)  [original NB05 split]',
        'train': (2003, 2005), 'test': (2006, 2009)
    },
]

strict_records = []
for sp in splits:
    tr_yrs = sp['train']; te_yrs = sp['test']
    # Use df_macro for A and B; use df_macro filtered to 2003+ for C (replicates original)
    data = df_macro
    tr_mask = data['year'].between(*tr_yrs)
    te_mask = data['year'].between(*te_yrs)
    X_tr = data.loc[tr_mask, M2_FEATURES]; y_tr = data.loc[tr_mask, TARGET]
    X_te = data.loc[te_mask, M2_FEATURES]; y_te = data.loc[te_mask, TARGET]

    print(f'Split {sp["name"]}:')
    print(f'  Train: {tr_mask.sum()} rows | {int(y_tr.sum())} positives')
    print(f'  Test : {te_mask.sum()} rows | {int(y_te.sum())} positives')

    if y_tr.sum() == 0 or y_te.sum() == 0:
        print('  Insufficient crisis cases — skip'); print(); continue

    m2 = RandomForestClassifier(n_estimators=500, max_depth=5,
                                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    m2.fit(X_tr, y_tr)
    prob = m2.predict_proba(X_te)[:, 1]

    auroc = roc_auc_score(y_te, prob); auprc = average_precision_score(y_te, prob)
    base  = y_te.mean()
    print(f'  AUROC={auroc:.4f}  AUPRC={auprc:.4f}  base={base:.4f}  AUPRC/base={auprc/base:.1f}x')
    print(f'  Literature AUROC benchmark: 0.75  |  {"≥ benchmark" if auroc >= 0.75 else "< benchmark"}')
    strict_records.append({'split': sp['name'], 'AUROC': auroc, 'AUPRC': auprc,
                            'base_rate': base, 'n_train_pos': int(y_tr.sum()),
                            'n_test_pos': int(y_te.sum())})
    print()

pd.DataFrame(strict_records).to_csv(ROB_DIR / 'augmented_strict_validation.csv', index=False)
print('Saved augmented_strict_validation.csv')

=== STAGE 1 STRICT TEMPORAL VALIDATION ===
Tests M2 trained on pre-crisis data against subsequent crisis episodes

Split A: Pre-Nordic (1988–1990) → Nordic+others (1991–1997):
  Train: 34 rows | 5 positives
  Test : 126 rows | 1 positives
  AUROC=0.5200  AUPRC=0.0164  base=0.0079  AUPRC/base=2.1x
  Literature AUROC benchmark: 0.75  |  < benchmark

Split B: Pre-GFC extended (1988–2002) → GFC (2003–2011):
  Train: 250 rows | 6 positives
  Test : 162 rows | 13 positives
  AUROC=0.4755  AUPRC=0.0762  base=0.0802  AUPRC/base=0.9x
  Literature AUROC benchmark: 0.75  |  < benchmark

Split C: Pre-GFC short (2003–2005) → GFC (2006–2009)  [original NB05 split]:
  Train: 54 rows | 0 positives
  Test : 72 rows | 13 positives
  Insufficient crisis cases — skip

Saved augmented_strict_validation.csv


## Cell 3 — Leave-one-country-out validation (Stage 1, extended panel)

In [7]:
print('=== LEAVE-ONE-COUNTRY-OUT VALIDATION (Stage 1: 1988–2020) ===')
print('Tests M2 cross-country generalisability on the extended panel.')
print()

loco_records = []
for ctry in sorted(df_macro['iso'].unique()):
    tr = df_macro[df_macro['iso'] != ctry]
    te = df_macro[df_macro['iso'] == ctry]
    X_tr = tr[M2_FEATURES]; y_tr = tr[TARGET]
    X_te = te[M2_FEATURES]; y_te = te[TARGET]
    if y_tr.sum() == 0 or y_te.sum() == 0: continue

    m2 = RandomForestClassifier(n_estimators=300, max_depth=5,
                                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    m2.fit(X_tr, y_tr)
    prob = m2.predict_proba(X_te)[:, 1]
    auroc = roc_auc_score(y_te, prob); auprc = average_precision_score(y_te, prob)
    print(f'  {ctry}: hold-out events={int(y_te.sum())}  AUROC={auroc:.4f}  AUPRC={auprc:.4f}')
    loco_records.append({'country': ctry, 'n_holdout': int(y_te.sum()),
                          'AUROC': auroc, 'AUPRC': auprc})

loco_df = pd.DataFrame(loco_records)
if len(loco_df) > 0:
    print()
    print(f'Mean AUROC across countries: {loco_df["AUROC"].mean():.4f}  std={loco_df["AUROC"].std():.4f}')
    print(f'Mean AUPRC across countries: {loco_df["AUPRC"].mean():.4f}  std={loco_df["AUPRC"].std():.4f}')
    loco_df.to_csv(ROB_DIR / 'augmented_loco_validation.csv', index=False)
    print('Saved augmented_loco_validation.csv')

=== LEAVE-ONE-COUNTRY-OUT VALIDATION (Stage 1: 1988–2020) ===
Tests M2 cross-country generalisability on the extended panel.

  BEL: hold-out events=1  AUROC=1.0000  AUPRC=1.0000
  CHE: hold-out events=2  AUROC=0.9833  AUPRC=0.8333
  DEU: hold-out events=1  AUROC=0.5484  AUPRC=0.0667
  DNK: hold-out events=1  AUROC=1.0000  AUPRC=1.0000
  ESP: hold-out events=1  AUROC=1.0000  AUPRC=1.0000
  FIN: hold-out events=1  AUROC=1.0000  AUPRC=1.0000
  FRA: hold-out events=1  AUROC=0.9677  AUPRC=0.5000
  GBR: hold-out events=2  AUROC=0.9000  AUPRC=0.3250
  IRL: hold-out events=1  AUROC=0.9032  AUPRC=0.2500
  ITA: hold-out events=2  AUROC=0.9000  AUPRC=0.6250
  JPN: hold-out events=1  AUROC=0.0690  AUPRC=0.0357
  NLD: hold-out events=1  AUROC=0.7419  AUPRC=0.1111
  PRT: hold-out events=1  AUROC=0.6552  AUPRC=0.0909
  SWE: hold-out events=2  AUROC=0.9500  AUPRC=0.7000
  USA: hold-out events=1  AUROC=0.9677  AUPRC=0.5000

Mean AUROC across countries: 0.8391  std=0.2539
Mean AUPRC across countries: 0

## Cell 4 — Alternative targets: H1 and H3 (Stage 2 dataset)

In [9]:
from sklearn.calibration import CalibratedClassifierCV

print('=== ALTERNATIVE PREDICTION HORIZONS (Stage 2: 2003–2020) ===')
print()

horizon_records = []
for hname, htarget in [('H1 (1-yr)', 'target_h1'), ('H2 (2-yr)', 'target_h2'),
                        ('H3 (3-yr)', 'target_h3')]:
    if htarget not in df_sent.columns: continue
    spw = (df_sent[htarget]==0).sum() / max((df_sent[htarget]==1).sum(), 1)
    X = df_sent[M3_FEATURES]; y = df_sent[htarget]
    if y.sum() == 0: continue

    # Simple single-fold train/test split for robustness check
    train_mask = df_sent['year'] <= 2015
    test_mask  = df_sent['year'] > 2015
    X_tr, y_tr = X[train_mask], y[train_mask]
    X_te, y_te = X[test_mask],  y[test_mask]

    if y_tr.sum() == 0 or y_te.sum() == 0:
        print(f'  {hname}: insufficient events in split — skip'); continue

    base_xgb = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
        scale_pos_weight=spw, random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)
    cal = CalibratedClassifierCV(estimator=base_xgb, method='isotonic', cv=2)
    try:
        cal.fit(X_tr, y_tr)
        prob = cal.predict_proba(X_te)[:, 1]
    except:
        base_xgb.fit(X_tr, y_tr)
        prob = base_xgb.predict_proba(X_te)[:, 1]

    auroc = roc_auc_score(y_te, prob); auprc = average_precision_score(y_te, prob)
    base  = y_te.mean()
    print(f'  {hname}: events={int(y_te.sum())}  AUROC={auroc:.4f}  AUPRC={auprc:.4f}  base={base:.4f}')
    horizon_records.append({'horizon': hname, 'AUROC': auroc, 'AUPRC': auprc,
                             'base_rate': base, 'n_events': int(y_te.sum())})

if horizon_records:
    pd.DataFrame(horizon_records).to_csv(ROB_DIR/'augmented_horizon_robustness.csv', index=False)
    print('\nSaved augmented_horizon_robustness.csv')

=== ALTERNATIVE PREDICTION HORIZONS (Stage 2: 2003–2020) ===

  H1 (1-yr): insufficient events in split — skip
  H2 (2-yr): insufficient events in split — skip
  H3 (3-yr): insufficient events in split — skip


## Cell 5 — Feature pruning check (Stage 2, M3)

In [11]:
from sklearn.feature_selection import SelectFromModel

print('=== FEATURE PRUNING CHECK (Stage 2: M3, 2003–2020) ===')
print(f'M3 features: {len(M3_FEATURES)}  |  Events: {int(df_sent[TARGET].sum())}')
print(f'Feature/event ratio: {len(M3_FEATURES)/max(int(df_sent[TARGET].sum()),1):.1f}x')
print()

X_full = df_sent[M3_FEATURES]; y_full = df_sent[TARGET]
spw = (y_full==0).sum() / max((y_full==1).sum(), 1)

xgb_sel = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=spw, random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)
xgb_sel.fit(X_full, y_full)

selector = SelectFromModel(xgb_sel, threshold='median', prefit=True)
M3_PRUNED = [f for f, keep in zip(M3_FEATURES, selector.get_support()) if keep]
print(f'Features after median pruning: {len(M3_PRUNED)} (removed {len(M3_FEATURES)-len(M3_PRUNED)})')

# Quick evaluation using 2016-2020 as test set
tr_m = df_sent['year'] <= 2015; te_m = df_sent['year'] > 2015
X_tr = df_sent.loc[tr_m, M3_PRUNED]; y_tr = df_sent.loc[tr_m, TARGET]
X_te = df_sent.loc[te_m, M3_PRUNED]; y_te = df_sent.loc[te_m, TARGET]

if y_tr.sum() > 0 and y_te.sum() > 0:
    spw_pr = (y_tr==0).sum() / max((y_tr==1).sum(), 1)
    m3_pr = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
        scale_pos_weight=spw_pr, random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)
    m3_pr.fit(X_tr, y_tr)
    prob_pr = m3_pr.predict_proba(X_te)[:, 1]

    auroc_pr = roc_auc_score(y_te, prob_pr); auprc_pr = average_precision_score(y_te, prob_pr)
    print(f'Pruned M3: AUROC={auroc_pr:.4f}  AUPRC={auprc_pr:.4f}')
    pd.DataFrame({'feature': M3_PRUNED}).to_csv(ROB_DIR/'augmented_M3_pruned_features.csv', index=False)
    print('Saved augmented_M3_pruned_features.csv')
else:
    print('Insufficient events for held-out evaluation.')

=== FEATURE PRUNING CHECK (Stage 2: M3, 2003–2020) ===
M3 features: 61  |  Events: 13
Feature/event ratio: 4.7x

Features after median pruning: 31 (removed 30)
Insufficient events for held-out evaluation.


In [12]:
print('=' * 65)
print(' NB08A AUGMENTED ROBUSTNESS CHECKS COMPLETE')
print('=' * 65)
print()
print('Key outputs:')
for f in sorted(ROB_DIR.glob('augmented_*.csv')):
    print(f'  {f.name}')
print()
print('Augmented pipeline complete.')
print('All results in:', AUG_DIR)

 NB08A AUGMENTED ROBUSTNESS CHECKS COMPLETE

Key outputs:
  augmented_loco_validation.csv
  augmented_strict_validation.csv

Augmented pipeline complete.
All results in: C:\Users\Owner\OneDrive\dissertation\data\processed\augmented_analysis
